# Chapter 14: Advanced Merging and Joining Strategies, Multi-Key Joins, and Performance Considerations

**Companion notebook** for *Beginner's Guide to Pandas* by Ravi Shankar

Run each cell in order. Exercises are at the end.

In [76]:
import pandas as pd
import numpy as np

# Advanced Merging and Joining Strategies, Multi-Key Joins, and Performance Considerations

## Introduction

As your datasets grow in complexity, you'll encounter situations where simple single-key joins aren't sufficient. This chapter explores sophisticated merging techniques, handling multiple join keys, and optimizing performance when working with large datasets.

## Understanding Join Types

Before diving into advanced techniques, let's clarify the four fundamental join types and when to use each one.

In [77]:
import pandas as pd
import numpy as np

# Create sample datasets
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'region': ['North', 'South', 'North', 'East']
})

orders = pd.DataFrame({
    'customer_id': [1, 2, 2, 5],
    'order_id': [101, 102, 103, 104],
    'amount': [150, 200, 175, 300]
})

### Inner Join

Returns only records that have matching keys in both DataFrames:

In [78]:
inner_result = customers.merge(orders, on='customer_id', how='inner')
# Result: 3 rows — customers 1 and 2 appear (customer 2 appears twice due to two orders)

### Left Join

Preserves all records from the left DataFrame, filling unmatched right-side columns with `NaN`:

In [79]:
left_result = customers.merge(orders, on='customer_id', how='left')
# Result: 4 rows — all customers appear; customers 3 and 4 have NaN for order columns

### Right Join

Preserves all records from the right DataFrame:

In [80]:
right_result = customers.merge(orders, on='customer_id', how='right')
# Result: 4 rows — all orders appear; customer_id 5 has NaN for name and region

### Outer Join

Preserves all records from both DataFrames:

In [81]:
outer_result = customers.merge(orders, on='customer_id', how='outer')
# Result: 5 rows — all customers (1–4) and the unmatched order (customer_id 5) appear

### Visual Comparison

```
Inner Join:        Left Join:          Right Join:         Outer Join:
┌─────┐           ┌─────┐             ┌─────┐             ┌─────┐
│  A  │           │  A  │             │  B  │             │  A  │
│  B  │ ┌─────┐   │  B  │ ┌─────┐     │  B  │ ┌─────┐     │  B  │ ┌─────┐
└─────┘ │  B  │   └─────┘ │  B  │     └─────┘ │  B  │     └─────┘ │  B  │
        │  C  │           │  C  │             │  C  │             │  C  │
        └─────┘           └─────┘             └─────┘             └─────┘
Result: B        Result: A,B,C       Result: B,C         Result: A,B,C
```

## Multi-Key Joins: Matching on Multiple Columns

When a single column isn't sufficient to uniquely identify records, multi-key joins become essential.

### Basic Multi-Key Join

In [82]:
# Sales data with region and product dimensions
sales_2023 = pd.DataFrame({
    'product': ['Widget', 'Widget', 'Gadget', 'Gadget'],
    'region': ['North', 'South', 'North', 'South'],
    'revenue_2023': [50000, 45000, 30000, 35000]
})

sales_2024 = pd.DataFrame({
    'product': ['Widget', 'Widget', 'Gadget', 'Gadget'],
    'region': ['North', 'South', 'North', 'South'],
    'revenue_2024': [55000, 48000, 32000, 38000]
})

# Join on both product and region
comparison = sales_2023.merge(
    sales_2024,
    on=['product', 'region'],
    how='inner'
)

Passing a list to `on` tells pandas to require a match on every listed column simultaneously. A row from `sales_2023` only joins with a row from `sales_2024` when both `product` and `region` match.

### Multi-Key Join with Different Column Names

When join keys have different names in each DataFrame, use `left_on` and `right_on`:

In [83]:
employees = pd.DataFrame({
    'emp_id': [1, 2, 3, 4],
    'dept_code': ['A', 'B', 'A', 'C'],
    'name': ['John', 'Jane', 'Jack', 'Jill'],
    'salary': [60000, 65000, 62000, 58000]
})

department_budgets = pd.DataFrame({
    'employee_id': [1, 2, 3, 4],
    'department': ['A', 'B', 'A', 'C'],
    'budget_allocation': [15000, 18000, 16000, 12000]
})

# Map different column names explicitly
result = employees.merge(
    department_budgets,
    left_on=['emp_id', 'dept_code'],
    right_on=['employee_id', 'department'],
    how='left'
)

# Clean up the now-redundant right-side key columns
result = result.drop(columns=['employee_id', 'department'])

### Multi-Key Join with a MultiIndex

For hierarchical data structures, you can join a regular DataFrame against a MultiIndex DataFrame:

In [84]:
# Create a MultiIndex DataFrame
index = pd.MultiIndex.from_tuples([
    ('North', 'Q1'),
    ('North', 'Q2'),
    ('South', 'Q1'),
    ('South', 'Q2')
], names=['region', 'quarter'])

quarterly_data = pd.DataFrame({
    'revenue': [50000, 55000, 45000, 48000]
}, index=index)

# Regular DataFrame with matching key columns
monthly_breakdown = pd.DataFrame({
    'region': ['North', 'North', 'South', 'South'],
    'quarter': ['Q1', 'Q2', 'Q1', 'Q2'],
    'detail': ['Jan-Mar', 'Apr-Jun', 'Jan-Mar', 'Apr-Jun']
})

# Join using columns from the left and the MultiIndex from the right
result = monthly_breakdown.merge(
    quarterly_data,
    left_on=['region', 'quarter'],
    right_index=True,
    how='left'
)

## Advanced Join Scenarios

### Joining on Columns vs. Indexes

The `.join()` method is an alternative to `.merge()` that operates on indexes by default, making it convenient when your DataFrames are already indexed on the join key.

In [85]:
# join() operates on indexes by default
df1 = pd.DataFrame({'A': [1, 2, 3]}, index=['a', 'b', 'c'])
df2 = pd.DataFrame({'B': [4, 5, 6]}, index=['a', 'b', 'c'])

result = df1.join(df2)  # Joins on index — no key column needed

You can also use `.join()` to match a column in the left DataFrame against the index of the right:

In [86]:
df1 = pd.DataFrame({
    'key': ['a', 'b', 'c'],
    'value1': [10, 20, 30]
})

df2 = pd.DataFrame({
    'value2': [100, 200, 300]
}, index=['a', 'b', 'c'])

result = df1.join(df2, on='key')  # Matches df1['key'] against df2.index

**When to use `.merge()` vs `.join()`:**

- Use `.merge()` when joining on columns, or when you need `left_on`/`right_on` for different column names.
- Use `.join()` when your DataFrames are already indexed on the join key, or when you want concise syntax for index-based joins.

In [87]:
# ----------------------------
# 2. merge() requires matching columns
# → so we convert df2 index into a column
# ----------------------------
df2_reset = df2.reset_index().rename(columns={'index': 'key'})

result_merge = df1.merge(df2_reset, on='key', how='inner')

print("\nMerge result:")
print(result_merge)

# ----------------------------
# 3. correct index-based join
# ----------------------------
df1_indexed = df1.set_index('key')
df2_indexed = df2  # already indexed correctly

result_index_join = df1_indexed.join(df2_indexed, how='inner')

print("\nIndex join result:")
print(result_index_join)


Merge result:
  key  value1  value2
0   a      10     100
1   b      20     200
2   c      30     300

Index join result:
     value1  value2
key                
a        10     100
b        20     200
c        30     300


### Handling Duplicate Column Names

When both DataFrames share column names that aren't part of the join key, use `suffixes` to distinguish them:

In [88]:
df1 = pd.DataFrame({
    'id': [1, 2, 3],
    'value': [10, 20, 30],
    'timestamp': ['2023-01-01', '2023-01-02', '2023-01-03']
})

df2 = pd.DataFrame({
    'id': [1, 2, 3],
    'value': [100, 200, 300],
    'timestamp': ['2023-01-01', '2023-01-02', '2023-01-03']
})

result = df1.merge(
    df2,
    on='id',
    how='inner',
    suffixes=('_source', '_target')
)
# Columns become: id, value_source, timestamp_source, value_target, timestamp_target

## Common Pitfalls and Solutions

### Pitfall 1: Unintended Cartesian Products

When multiple rows in both DataFrames share the same key, pandas creates every possible combination — which can cause row counts to explode unexpectedly.

In [89]:
df1 = pd.DataFrame({'key': [1, 1, 2], 'value1': ['a', 'b', 'c']})
df2 = pd.DataFrame({
    'key': [1, 2],
    'value2': ['x', 'z']
})
result = df1.merge(df2, on='key')
# Result has 5 rows instead of 3:
# key=1 has 2 rows in df1 and 2 rows in df2 → 4 combinations
# key=2 has 1 row in each → 1 combination

**Solution — aggregate before joining:**

In [90]:
# Summarise one DataFrame so keys are unique before merging
df1_summary = df1.groupby('key')['value1'].agg(list).reset_index()
result = df1_summary.merge(df2, on='key', how='left')

**Solution — use `validate` to catch the problem early:**

In [91]:
# Raises MergeError if df2 contains duplicate keys
result = df1.merge(df2, on='key', how='left', validate='m:1')

**Solution — check for duplicates beforehand:**

In [92]:
duplicates = df2[df2.duplicated(subset=['key'], keep=False)]
if not duplicates.empty:
    print("Warning: Duplicate keys found in df2")

### Pitfall 2: Data Type Mismatches

If the join key has different types in each DataFrame, pandas cannot match the values and returns an empty result — with no error or warning.

In [93]:
import pandas as pd

df1 = pd.DataFrame({'id': [1, 2, 3]})
df2 = pd.DataFrame({'id': ['1', '2', '3']})

df2['id'] = df2['id'].astype(int)

result = df1.merge(df2, on='id')

print(result)

   id
0   1
1   2
2   3


**Solution:** Ensure consistent data types before merging:

In [94]:
df2['id'] = df2['id'].astype('int')
result = df1.merge(df2, on='id')

### Pitfall 3: NaN Values in Join Keys

`NaN` values never match each other in a join, so rows with `NaN` keys are silently dropped.

In [95]:
df1 = pd.DataFrame({'key': [1, 2, np.nan], 'value1': ['a', 'b', 'c']})
df2 = pd.DataFrame({'key': [1, 2, np.nan], 'value2': ['x', 'y', 'z']})

result = df1.merge(df2, on='key')
# NaN rows don't match — result has only 2 rows

**Solution:** Replace `NaN` with a sentinel value before joining:

In [96]:
df1['key'] = df1['key'].fillna(-999)
df2['key'] = df2['key'].fillna(-999)
result = df1.merge(df2, on='key')

## Performance Optimization Strategies

### 1. Filter Before Joining

Reducing dataset size before a merge is the single most impactful optimization. Smaller inputs mean less work at every stage.

```python
# Inefficient: merge first, then filter
result = df1.merge(df2, on='key', how='inner')
result = result[result['year'] == 2024]


# ----------------------------
# Better: filter before merge
# ----------------------------
df1_filtered = df1[df1['year'] == 2024]
df2_filtered = df2[df2['year'] == 2024]

result = df1_filtered.merge(df2_filtered, on='key', how='inner')
```

### 2. Use Appropriate Data Types

Numeric keys are faster to hash and compare than string keys, and smaller integer types reduce memory pressure.

In [ ]:
# Create a large DataFrame
df1 = pd.DataFrame({
    'id': np.arange(1_000_000),
    'category': np.random.choice(['A', 'B', 'C'], 1_000_000)
})

# Downcast to smaller types before merging
df1['id'] = df1['id'].astype('int32')          # Instead of int64
df1['category'] = df1['category'].astype('category')

### 3. Use Index-Based Joins for Repeated Merges

Setting the join key as the index once and reusing it avoids repeated hashing of the same column.

```python
# Set index on the join key once
df_indexed = df2.set_index('customer_id')

# Reuse the indexed DataFrame for multiple joins
result1 = df_a.merge(df_indexed, left_on='cust_id', right_index=True, how='left')
result2 = df_b.merge(df_indexed, left_on='cust_id', right_index=True, how='left')
```

### 4. Pre-Sort Before Joining Large Datasets

For very large DataFrames, sorting on the join key before merging can improve cache locality and reduce the cost of the merge algorithm.

In [ ]:
df1_sorted = df1.sort_values('key').reset_index(drop=True)
df2_sorted = df2.sort_values('key').reset_index(drop=True)

result = df1_sorted.merge(df2_sorted, on='key', how='left')

### 5. Monitor Memory Usage

Before joining large DataFrames, check their sizes so you can anticipate whether the result will fit in memory.

In [ ]:
size_df1 = df1.memory_usage(deep=True).sum() / 1024**2  # MB
size_df2 = df2.memory_usage(deep=True).sum() / 1024**2

print(f"df1 size: {size_df1:.2f} MB")
print(f"df2 size: {size_df2:.2f} MB")

result = df1.merge(df2, on='key', how='inner')

size_result = result.memory_usage(deep=True).sum() / 1024**2
print(f"Result size: {size_result:.2f} MB")

### Performance Comparison: Column-Based vs. Index-Based Merge

In [ ]:
import time

# Create large datasets
n = 1_000_000
df_large1 = pd.DataFrame({
    'key': np.random.randint(0, 100_000, n),
    'value1': np.random.randn(n)
})

df_large2 = pd.DataFrame({
    'key': np.arange(100_000),
    'value2': np.random.randn(100_000)
})

# Method 1: Direct column-based merge
start = time.time()
result1 = df_large1.merge(df_large2, on='key', how='left')
time1 = time.time() - start

# Method 2: Index-based merge
start = time.time()
df_large2_indexed = df_large2.set_index('key')
result2 = df_large1.merge(df_large2_indexed, left_on='key', right_index=True, how='left')
time2 = time.time() - start

print(f"Direct merge:      {time1:.4f}s")
print(f"Index-based merge: {time2:.4f}s")

## Real-World Example: Complex Multi-Step Join

The following example ties together everything covered in this chapter. We join four DataFrames in sequence, then compute business metrics on the combined result.

In [ ]:
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'region_code': ['R1', 'R2', 'R1', 'R3', 'R2'],
    'segment': ['Premium', 'Standard', 'Premium', 'Standard', 'Premium']
})

transactions = pd.DataFrame({
    'transaction_id': [101, 102, 103, 104, 105, 106],
    'customer_id': [1, 1, 2, 3, 4, 5],
    'product_id': ['P1', 'P2', 'P1', 'P3', 'P2', 'P1'],
    'amount': [100, 150, 200, 75, 125, 180]
})

products = pd.DataFrame({
    'product_id': ['P1', 'P2', 'P3'],
    'category': ['Electronics', 'Home', 'Electronics'],
    'margin': [0.30, 0.25, 0.35]
})

regions = pd.DataFrame({
    'region_code': ['R1', 'R2', 'R3'],
    'region_name': ['North', 'South', 'East'],
    'tax_rate': [0.08, 0.07, 0.09]
})

# Step 1: Join transactions with customer info
step1 = transactions.merge(customers, on='customer_id', how='left')

# Step 2: Join with product details
step2 = step1.merge(products, on='product_id', how='left')

# Step 3: Join with regional information
final_analysis = step2.merge(regions, on='region_code', how='left')

# Calculate business metrics
final_analysis['profit'] = final_analysis['amount'] * final_analysis['margin']
final_analysis['tax'] = final_analysis['amount'] * final_analysis['tax_rate']
final_analysis['net_revenue'] = final_analysis['amount'] - final_analysis['tax']

# Summarise by customer segment and region
summary = final_analysis.groupby(['segment', 'region_name']).agg(
    total_amount=('amount', 'sum'),
    total_profit=('profit', 'sum'),
    transaction_count=('transaction_id', 'count')
)

Each step uses a left join so that no transactions are lost even if a lookup table is missing a matching key — a safe default for analytical pipelines.

## Summary

Advanced merging strategies enable you to work with complex, multi-dimensional data relationships. Key takeaways:

- **Choose the right join type**: inner, left, right, and outer joins serve different purposes — pick based on which records you need to retain.
- **Multi-key joins**: pass a list to `on`, or use `left_on`/`right_on`, when a single column isn't sufficient for unique matching.
- **Watch for pitfalls**: unintended Cartesian products, data type mismatches, and `NaN` keys are the most common sources of silent errors.
- **Optimize deliberately**: filter early, use appropriate data types, and consider index-based joins for repeated or large-scale merges.
- **Validate your results**: always check row counts and sample the output after joining to confirm the result matches your expectations.

---

# Exercises

Test your understanding of this chapter's concepts.

### Exercise 1: Mastering Join Types

Practice the four fundamental join types (inner, left, right, outer) using two small DataFrames representing employees and departments. Observe how each join type affects the resulting rows and missing values.

In [ ]:
import pandas as pd

employees = pd.DataFrame({
    'emp_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'dept_id': [10, 20, 10, 30, 99]
})

departments = pd.DataFrame({
    'dept_id': [10, 20, 30, 40],
    'dept_name': ['Engineering', 'Marketing', 'Finance', 'HR']
})

# TODO: Perform an inner join on 'dept_id' and store in inner_join
inner_join = None

# TODO: Perform a left join on 'dept_id' and store in left_join
left_join = None

# TODO: Perform a right join on 'dept_id' and store in right_join
right_join = None

# TODO: Perform a full outer join on 'dept_id' and store in outer_join
outer_join = None

# TODO: Print the number of rows in each result and inspect outer_join
print('Inner:', None)
print('Left:', None)
print('Right:', None)
print('Outer:', None)
print('\nOuter join result:')
print(outer_join)

### Exercise 2: Multi-Key Joins

Join two DataFrames representing sales orders and product pricing using multiple columns as keys. This simulates a real scenario where a unique record is identified by a combination of fields rather than a single column.

In [ ]:
import pandas as pd

orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105],
    'product': ['Widget', 'Gadget', 'Widget', 'Doohickey', 'Gadget'],
    'region': ['North', 'South', 'South', 'North', 'North'],
    'quantity': [5, 3, 8, 2, 6]
})

pricing = pd.DataFrame({
    'product': ['Widget', 'Widget', 'Gadget', 'Gadget', 'Doohickey'],
    'region': ['North', 'South', 'North', 'South', 'North'],
    'unit_price': [10.0, 9.5, 25.0, 24.0, 15.0]
})

# TODO: Merge orders with pricing using BOTH 'product' and 'region' as keys
# Use an inner join and store the result in merged
merged = None

# TODO: Add a new column 'total_price' = quantity * unit_price
merged['total_price'] = None

# TODO: Print the merged DataFrame sorted by 'total_price' descending
print(None)

### Exercise 3: Handling Duplicate Columns and Suffixes

Merge two DataFrames that share a non-key column name, causing a naming conflict. Resolve the conflict using custom suffixes and then clean up the resulting DataFrame by dropping or renaming columns as needed.

In [ ]:
import pandas as pd

product_info = pd.DataFrame({
    'product_id': [1, 2, 3, 4],
    'product_name': ['Alpha', 'Beta', 'Gamma', 'Delta'],
    'last_updated': ['2023-01-10', '2023-03-15', '2022-11-01', '2023-06-20']
})

warehouse_stock = pd.DataFrame({
    'product_id': [1, 2, 3, 5],
    'warehouse': ['WH-A', 'WH-B', 'WH-A', 'WH-C'],
    'last_updated': ['2023-01-12', '2023-03-10', '2022-11-05', '2023-07-01']
})

# TODO: Merge the two DataFrames on 'product_id' using a left join.
# Use suffixes '_info' and '_warehouse' to distinguish the shared 'last_updated' column.
merged = None

# TODO: Rename 'last_updated_info' to 'info_date' and
# 'last_updated_warehouse' to 'stock_date'
merged = merged.rename(columns=None)

# TODO: Print the final DataFrame
print(merged)

### Exercise 4: Multi-Step Join with Performance Optimization

Perform a multi-step join pipeline across three DataFrames: customers, transactions, and store locations. Apply performance best practices by filtering and selecting only necessary columns before merging, then validate the final result.

In [ ]:
import pandas as pd

customers = pd.DataFrame({
    'customer_id': range(1, 7),
    'name': ['Ana', 'Ben', 'Cara', 'Dan', 'Ella', 'Frank'],
    'tier': ['Gold', 'Silver', 'Gold', 'Bronze', 'Silver', 'Bronze']
})

transactions = pd.DataFrame({
    'txn_id': range(201, 210),
    'customer_id': [1, 2, 1, 3, 5, 2, 4, 1, 6],
    'store_id': [10, 20, 10, 30, 20, 30, 10, 20, 30],
    'amount': [120, 45, 200, 89, 300, 60, 150, 95, 40]
})

stores = pd.DataFrame({
    'store_id': [10, 20, 30],
    'city': ['New York', 'Chicago', 'Houston'],
    'region': ['East', 'Midwest', 'South']
})

# TODO: For performance, select only needed columns from each DataFrame before merging.
# From customers keep: 'customer_id', 'name', 'tier'
# From transactions keep: 'txn_id', 'customer_id', 'store_id', 'amount'
# From stores keep: 'store_id', 'city'
customers_slim = None
transactions_slim = None
stores_slim = None

# TODO: Step 1 — merge transactions_slim with customers_slim on 'customer_id' (inner join)
step1 = None

# TODO: Step 2 — merge step1 with stores_slim on 'store_id' (inner join)
final = None

# TODO: Filter to only 'Gold' tier customers and print the total amount spent by city
gold = None
print(None)

---

# Solutions

*Scroll down only after you've attempted the exercises above.*

<br><br><br><br><br><br><br><br><br><br>

### Solution 1: Mastering Join Types

In [ ]:
import pandas as pd
import numpy as np

employees = pd.DataFrame({
    'emp_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'dept_id': [10, 20, 10, 30, 99]
})

departments = pd.DataFrame({
    'dept_id': [10, 20, 30, 40],
    'dept_name': ['Engineering', 'Marketing', 'Finance', 'HR']
})

# Perform an inner join on 'dept_id'
inner_join = pd.merge(employees, departments, on='dept_id', how='inner')

# Perform a left join on 'dept_id'
left_join = pd.merge(employees, departments, on='dept_id', how='left')

# Perform a right join on 'dept_id'
right_join = pd.merge(employees, departments, on='dept_id', how='right')

# Perform a full outer join on 'dept_id'
outer_join = pd.merge(employees, departments, on='dept_id', how='outer')

# Print the number of rows in each result and inspect outer_join
print('Inner:', len(inner_join))
print('Left:', len(left_join))
print('Right:', len(right_join))
print('Outer:', len(outer_join))
print('\nOuter join result:')
print(outer_join)

### Solution 2: Multi-Key Joins

In [ ]:
import pandas as pd
import numpy as np

orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105],
    'product': ['Widget', 'Gadget', 'Widget', 'Doohickey', 'Gadget'],
    'region': ['North', 'South', 'South', 'North', 'North'],
    'quantity': [5, 3, 8, 2, 6]
})

pricing = pd.DataFrame({
    'product': ['Widget', 'Widget', 'Gadget', 'Gadget', 'Doohickey'],
    'region': ['North', 'South', 'North', 'South', 'North'],
    'unit_price': [10.0, 9.5, 25.0, 24.0, 15.0]
})

# Merge orders with pricing using BOTH 'product' and 'region' as keys
merged = pd.merge(orders, pricing, on=['product', 'region'], how='inner')

# Add a new column 'total_price' = quantity * unit_price
merged['total_price'] = merged['quantity'] * merged['unit_price']

# Print the merged DataFrame sorted by 'total_price' descending
print(merged.sort_values('total_price', ascending=False))

### Solution 3: Handling Duplicate Columns and Suffixes

In [ ]:
import pandas as pd
import numpy as np

product_info = pd.DataFrame({
    'product_id': [1, 2, 3, 4],
    'product_name': ['Alpha', 'Beta', 'Gamma', 'Delta'],
    'last_updated': ['2023-01-10', '2023-03-15', '2022-11-01', '2023-06-20']
})

warehouse_stock = pd.DataFrame({
    'product_id': [1, 2, 3, 5],
    'warehouse': ['WH-A', 'WH-B', 'WH-A', 'WH-C'],
    'last_updated': ['2023-01-12', '2023-03-10', '2022-11-05', '2023-07-01']
})

# Merge the two DataFrames on 'product_id' using a left join.
# Use suffixes '_info' and '_warehouse' to distinguish the shared 'last_updated' column.
merged = pd.merge(
    product_info,
    warehouse_stock,
    on='product_id',
    how='left',
    suffixes=('_info', '_warehouse')
)

# Rename 'last_updated_info' to 'info_date' and 'last_updated_warehouse' to 'stock_date'
merged = merged.rename(columns={
    'last_updated_info': 'info_date',
    'last_updated_warehouse': 'stock_date'
})

# Print the final DataFrame
print(merged)

### Solution 4: Multi-Step Join with Performance Optimization

In [ ]:
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    'customer_id': range(1, 7),
    'name': ['Ana', 'Ben', 'Cara', 'Dan', 'Ella', 'Frank'],
    'tier': ['Gold', 'Silver', 'Gold', 'Bronze', 'Silver', 'Bronze']
})

transactions = pd.DataFrame({
    'txn_id': range(201, 210),
    'customer_id': [1, 2, 1, 3, 5, 2, 4, 1, 6],
    'store_id': [10, 20, 10, 30, 20, 30, 10, 20, 30],
    'amount': [120, 45, 200, 89, 300, 60, 150, 95, 40]
})

stores = pd.DataFrame({
    'store_id': [10, 20, 30],
    'city': ['New York', 'Chicago', 'Houston'],
    'region': ['East', 'Midwest', 'South']
})

# For performance, select only needed columns before merging
customers_slim = customers[['customer_id', 'name', 'tier']]
transactions_slim = transactions[['txn_id', 'customer_id', 'store_id', 'amount']]
stores_slim = stores[['store_id', 'city']]

# Step 1 — merge transactions_slim with customers_slim on 'customer_id'
step1 = pd.merge(transactions_slim, customers_slim, on='customer_id', how='inner')

# Step 2 — merge step1 with stores_slim on 'store_id'
final = pd.merge(step1, stores_slim, on='store_id', how='inner')

# Filter to only 'Gold' tier customers and print total amount spent by city
gold = final[final['tier'] == 'Gold']
print(gold.groupby('city')['amount'].sum())